# Spec-FastGS — Mip-NeRF 360 Dataset Sweep & Inference Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full Spec-FastGS training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment with configurable image resolution (`images_4`, `images_2`, `images`, `images_8`).

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables.
2. **Dataset Verification**: Verifies the presence of the Mip-NeRF 360 dataset root `/home/ghp4hc/datasets/datasets/mipneft360`.
3. **Resolution & Layout Validation**: Verifies selected resolution (`IMAGES`) exists for all 9 scenes (`bicycle`, `flowers`, `garden`, `stump`, `treehill`, `room`, `counter`, `kitchen`, `bonsai`).
4. **Run Sweep**: Invokes `run_mip360.sh` with `DATA_ROOT="/home/ghp4hc/datasets/datasets/mipneft360"` and `IMAGES` variable, logging outputs to `mip360_{IMAGES}_run.log`.
5. **Results Aggregation**: Formats and prints quantitative metrics (`results_grouped.json`) in a neat table.
6. **Output Archiving**: Zips the results to `spec_fastgs_output_mip360_{IMAGES}.zip` in the parent directory.
7. **Hugging Face Upload**: Automatically uploads the zipped output results back to `DiBiay/spec-fastgs-mipnerf360-result` under `images4.zip`, `images2.zip`, `images.zip`, or `images8.zip`.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config, Resolution & Kernel Check
Defines paths, image resolution choice (`images_4`, `images_2`, `images`, `images_8`), and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Image resolution setting: 'images_4' (default), 'images_2', 'images', or 'images_8'
IMAGES = os.environ.get('IMAGES', 'images_4')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

ENV_NAME = 'thesis_env'

print(f'Active Python      : {sys.executable}')
print(f'Active Kernel name : {ENV_NAME}')
print(f'Repository Root    : {REPO_ROOT}')
print(f'Selected Resolution: {IMAGES}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiled custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Mip-NeRF 360 Dataset Layout
Validates the subdirectories to ensure all scenes and `{IMAGES}` exist.

In [ ]:
# ── Verify Mip-NeRF 360 Dataset Layout (all 9 scenes) ───────────────────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]
IMAGES = os.environ.get('IMAGES', 'images_4')

print(f"Verifying layouts for resolution '{IMAGES}' directly under {src_root} ...")
missing = []
for scene in MIP360_SCENES:
    v2_path = os.path.join(src_root, "360_v2", scene)
    extra_path = os.path.join(src_root, "360_extra_scenes", scene)
    
    if os.path.isdir(v2_path):
        scene_dir = v2_path
    elif os.path.isdir(extra_path):
        scene_dir = extra_path
    else:
        scene_dir = None
        
    if scene_dir is None:
        status = "MISSING (scene folder not found)"
        missing.append(scene)
    else:
        images_dir = os.path.join(scene_dir, IMAGES)
        if not os.path.isdir(images_dir):
            status = f"MISSING ({IMAGES} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
            missing.append(scene)
        else:
            n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            status = f"OK ({n_imgs} images)"
            
    print(f"  {scene:<12s} {status}")

print()
if missing:
    print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {IMAGES}: {missing}")
    print("    run_mip360.sh will skip these scenes during the sweep.")
else:
    print(f"✅ All {len(MIP360_SCENES)} scenes are verified for '{IMAGES}'. Ready for training!")

## c05 — Run Spec-FastGS Sweep (`run_mip360.sh`)
Launches the training sweep using the `thesis_env` virtual environment, system CUDA compiler paths, and selected `IMAGES` resolution.

In [ ]:
# ── Prepare script inputs ──────────────────────────────────────────────────────
import subprocess
import os
import sys

IMAGES = os.environ.get('IMAGES', 'images_4')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

LOGFILE = os.path.join(os.path.dirname(REPO_ROOT), f"mip360_{IMAGES}_specfastgs_run.log")
DATA_ROOT = "/home/ghp4hc/datasets/datasets/mipneft360"

venv_bin = os.path.dirname(sys.executable)
print(f"Virtual environment bin path: {venv_bin}")

cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home:
    # Auto-load search script
    search_script = 'which nvcc 2>/dev/null || (for init in /etc/profile /etc/profile.d/modules.sh; do [ -f "$init" ] && source "$init"; done && for mod in cuda/11.7 cuda/11.8 cuda/12.1 cuda/12.6; do module load "$mod" 2>/dev/null; done && which nvcc 2>/dev/null)'
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        cuda_home = '/usr/local/cuda'

print(f"CUDA_HOME path: {cuda_home}")

In [ ]:
%%bash -s "$venv_bin" "$cuda_home" "$LOGFILE" "$REPO_ROOT" "$DATA_ROOT" "$IMAGES"
ENV_BIN=$1
CUDA_HOME=$2
LOGFILE=$3
REPO_ROOT=$4
export DATA_ROOT=$5
export IMAGES=$6

export PATH=$ENV_BIN:$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd "$REPO_ROOT"

echo "Running run_mip360.sh (all 9 scenes directly from ${DATA_ROOT}, ${IMAGES}) ..."
bash run_mip360.sh > "$LOGFILE" 2>&1
STATUS=$?

echo "--- tail of ${LOGFILE} ---"
tail -n 100 "$LOGFILE"
echo "run_mip360.sh exit status: $STATUS"
if [ $STATUS -ne 0 ]; then
    echo "⚠️  run_mip360.sh stopped early (STOP_ON_ERROR=True) -- check logs above or at ${LOGFILE}"
fi

## c06 — Quantitative Results Summary
Parses `results_grouped.json` and `train_info.json` from the output directory to print a formatted metrics table.

In [ ]:
# ── Quantitative Results Summary ──────────────────────────────────────────────
import json
import os

IMAGES = os.environ.get('IMAGES', 'images_4')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

OUTPUT_ROOT = os.path.join(REPO_ROOT, "output", f"mip360_{IMAGES}")

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

header = f"{'scene':<12s}{'PSNR':>8s}{'SSIM':>8s}{'LPIPS':>8s}{'Spec_PSNR':>11s}{'ASG_IoU':>9s}{'#Gauss':>10s}{'time':>10s}"
print(header)
print("-" * len(header))
for scene in MIP360_SCENES:
    out_dir = os.path.join(OUTPUT_ROOT, scene)
    results_path = os.path.join(out_dir, "results_grouped.json")
    info_path = os.path.join(out_dir, "train_info.json")

    if not os.path.exists(results_path):
        print(f"{scene:<12s}  (no results_grouped.json -- skipped, or sweep did not reach this scene)")
        continue

    with open(results_path) as f:
        results = json.load(f)
    scene_result = next(iter(results.values()))
    render_result = next(iter(scene_result.values()))
    main = render_result.get("main_metrics", {})
    aux = render_result.get("aux_metrics", {})

    info = {}
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = json.load(f)

    print(f"{scene:<12s}{fmt(main.get('PSNR')):>8s}{fmt(main.get('SSIM')):>8s}{fmt(main.get('LPIPS')):>8s}"
          f"{fmt(aux.get('Spec_PSNR')):>11s}{fmt(aux.get('ASG_Residual_IoU')):>9s}"
          f"{str(info.get('final_gaussians', '-')):>10s}{str(info.get('training_time_formatted', '-')):>10s}")

## c07 — Packaging Submission ZIP
Zips the outputs directory into `spec_fastgs_output_mip360_{IMAGES}.zip`.

In [ ]:
# ── Packaging Submission ZIP ──────────────────────────────────────────────────
import shutil
import os

IMAGES = os.environ.get('IMAGES', 'images_4')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

# Try to find the output folder to zip
candidates = [
    f"/home/ghp4hc/thesis-all/spec_fastgs_output_mip360_{IMAGES}",
    os.path.join(REPO_ROOT, "output", f"mip360_{IMAGES}"),
    os.path.join(os.path.dirname(REPO_ROOT), f"spec_fastgs_output_mip360_{IMAGES}"),
]

src = None
for c in candidates:
    if os.path.isdir(c):
        src = c
        break

out = os.path.join(os.path.dirname(REPO_ROOT), f"spec_fastgs_output_mip360_{IMAGES}")

if src:
    print(f"📦 Zipping folder '{src}' -> '{out}.zip' ...")
    # Remove existing zip if exists to ensure clean write
    if os.path.exists(out + ".zip"):
        os.remove(out + ".zip")
    shutil.make_archive(out, "zip", src)
    print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
else:
    print("❌ ERROR: Output folder not found in any of the candidate paths:", candidates)

## c08 — Hugging Face Results Upload
Uploads the zipped results archive back to `DiBiay/spec-fastgs-mipnerf360-result` under `images4.zip`, `images2.zip`, `images.zip`, or `images8.zip`.

In [ ]:
# ── Upload Results Zip to Hugging Face ─────────────────────────────────────────
import os
import sys
import subprocess

try:
    from huggingface_hub import HfApi
except ImportError:
    print("huggingface_hub not found. Installing via pip...")
    PROXY = 'http://rb-proxy-sl.bosch.com:8080'
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    from huggingface_hub import HfApi

IMAGES = os.environ.get('IMAGES', 'images_4')

# Map resolution option to target zip filename in Hugging Face repository
zip_name_map = {
    'images_4': 'images4.zip',
    'images_2': 'images2.zip',
    'images':   'images.zip',
    'images_8': 'images8.zip',
}
target_repo_filename = zip_name_map.get(IMAGES, f"{IMAGES}.zip")

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

# Load Hugging Face credentials & repository target
HF_TOKEN = os.environ.get('HF_TOKEN', '<YOUR_HF_TOKEN>')
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/spec-fastgs-mipnerf360-result')

# Try to find the zip archive file
zip_candidates = [ 
    os.path.join(os.path.dirname(REPO_ROOT), f'spec_fastgs_output_mip360_{IMAGES}.zip'),
    os.path.join(os.getcwd(), f'spec_fastgs_output_mip360_{IMAGES}.zip'),
    f'/home/ghp4hc/thesis-all/spec_fastgs_output_mip360_{IMAGES}.zip',
    f'/home/ghp4hc/spec_fastgs_output_mip360_{IMAGES}.zip',
]
zip_path = None
for z in zip_candidates:
    if os.path.isfile(z):
        zip_path = z
        break

if HF_TOKEN == '<YOUR_HF_TOKEN>' or not HF_TOKEN:
    print("⚠️  Hugging Face token not configured in environment variable 'HF_TOKEN'.")
    HF_TOKEN = input("Enter your Hugging Face Access Token: ").strip()

if zip_path and HF_TOKEN and HF_TOKEN != '<YOUR_HF_TOKEN>':
    print(f"🔍 Checking repository '{HF_REPO}' status...")
    api = HfApi()
    
    # Auto-create repository if it does not exist
    try:
        api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print(f"✅ Repository '{HF_REPO}' already exists.")
    except Exception:
        print(f"➕ Creating private dataset repository '{HF_REPO}'...")
        try:
            api.create_repo(
                repo_id=HF_REPO,
                repo_type="dataset",
                token=HF_TOKEN,
                private=True,
            )
            print(f"🎉 Created private repository '{HF_REPO}' successfully.")
        except Exception as e:
            print(f"⚠️  Could not create repository: {e}")

    print(f"📤 Uploading '{zip_path}' as '{target_repo_filename}' to dataset repository '{HF_REPO}'...")
    try:
        api.upload_file(
            path_or_fileobj=zip_path,
            path_in_repo=target_repo_filename,
            repo_id=HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print(f"🎉 [SUCCESS] Results archive '{target_repo_filename}' uploaded to Hugging Face repository '{HF_REPO}' successfully!")
    except Exception as e:
        print(f"❌ [ERROR] Hugging Face upload failed: {e}")
else:
    print(f"⚠️  Skipping upload: zip file not found or invalid HF_TOKEN.")
    if not zip_path:
        print(f"    Looked in paths: {zip_candidates}")